In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from datetime import datetime

plt.rcParams.update({
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

In [ ]:
for f in sorted(Path(".").glob("*.json")):
    try:
        prefix, date_part, time_part = f.stem.split("_")[0].split("-")
        dt = datetime.strptime(date_part + time_part, "%y%m%d%H%M%S")
        label = dt.strftime("%Y-%m-%d %H:%M")
    except ValueError:
        label = ""
    print(f'"{f.name}"{"  # " + label if label else ""}')

In [ ]:
_exports = sorted(Path(".").glob("*_export.json"))
if not _exports:
    raise FileNotFoundError("No *_export.json file found next to the notebook. Drop an export here and re-run.")
JSON_PATH = _exports[0]
print(f"Loading {JSON_PATH.name}")

with open(JSON_PATH) as f:
    raw = json.load(f)

test  = raw["test"]
stats = test["statistics"]

In [ ]:
all_sample_t = [s["t"] for rec in test["recordings"] for s in rec["data"]]
all_suds_t   = [e["t"] for e in test.get("sudsEvents", [])]
t0_ms        = min(all_sample_t + all_suds_t) if (all_sample_t or all_suds_t) else 0

def elapsed(ts_ms):
    return (ts_ms - t0_ms) / 1000.0

rows = []
for rec in test["recordings"]:
    for s in rec["data"]:
        rows.append({
            "recording_seq": rec["sequence"],
            "elapsed_s":     elapsed(s["t"]),
            "value":         s["v"],
            "sensor":        s["s"],
        })
df = pd.DataFrame(rows) if rows else pd.DataFrame(
    columns=["recording_seq", "elapsed_s", "value", "sensor"]
)

# (sensor key, subplot title, y-axis label, line color). Order = subplot order.
SENSOR_SPECS = [
    ("esensePulse_hr",  "eSense Pulse — Heart Rate",  "BPM",         "#e53935"),
    ("fibion_hr",       "Fibion — Heart Rate",        "BPM",         "#d81b60"),
    ("esensePulse_rr",  "eSense Pulse — RR Interval", "RR (ms)",     "#fb8c00"),
    ("fibion_rr",       "Fibion — RR Interval",       "RR (ms)",     "#f4511e"),
    ("esenseResp_resp", "Respiration Signal",         "Respiration", "#1e88e5"),
    ("fibion_ecg",      "Fibion — ECG",               "ECG (mV)",    "#3949ab"),
]
sensor_frames = {key: df[df["sensor"] == key] for key, *_ in SENSOR_SPECS}

rec_windows = []
for rec in test["recordings"]:
    ts = [s["t"] for s in rec["data"]]
    start_ms, end_ms = (min(ts), max(ts)) if ts else (t0_ms, t0_ms + rec["durationMs"])
    rec_windows.append({"sequence": rec["sequence"], "start_s": elapsed(start_ms), "end_s": elapsed(end_ms)})
df_rec = pd.DataFrame(rec_windows)

df_suds = (
    pd.DataFrame([{"elapsed_s": elapsed(e["t"]), "value": e["v"]} for e in test["sudsEvents"]])
    if test.get("sudsEvents")
    else pd.DataFrame(columns=["elapsed_s", "value"])
)

In [ ]:
SUDS_COLOR = "#7b1fa2"

def shade_recordings(ax, df_rec):
    for _, row in df_rec.iterrows():
        ax.axvspan(row["start_s"], row["end_s"], alpha=0.10, color="#4caf50", zorder=0)

def mark_suds(ax, df_suds, y_min, y_max):
    for _, row in df_suds.iterrows():
        ax.axvline(row["elapsed_s"], color=SUDS_COLOR, linestyle="--", linewidth=1.2, alpha=0.8)
        ax.text(
            row["elapsed_s"] + (y_max - y_min) * 0.003,
            y_min + (y_max - y_min) * 0.97,
            f"SUDs {int(row['value'])}",
            color=SUDS_COLOR, fontsize=8, va="top", rotation=90,
        )

# Render one subplot per sensor type present in the export.
present = [(key, title, ylabel, color)
           for (key, title, ylabel, color) in SENSOR_SPECS
           if not sensor_frames[key].empty]

if not present:
    print("No sensor data in this session.")
else:
    fig, axes = plt.subplots(
        len(present), 1,
        figsize=(14, 3.5 * len(present)),
        sharex=True,
        constrained_layout=True,
    )
    if len(present) == 1:
        axes = [axes]

    legend_handles = [
        mpatches.Patch(color="#4caf50", alpha=0.3, label="Recording window"),
        plt.Line2D([0], [0], color=SUDS_COLOR, linestyle="--", label="SUDs event"),
    ]

    for ax, (key, title, ylabel, color) in zip(axes, present):
        sdf = sensor_frames[key]
        ax.plot(sdf["elapsed_s"], sdf["value"], color=color, linewidth=1.2)
        shade_recordings(ax, df_rec)
        if not df_suds.empty:
            mark_suds(ax, df_suds, sdf["value"].min(), sdf["value"].max())
        ax.set_ylabel(ylabel)
        ax.set_title(title, color=color)
        ax.legend(handles=legend_handles, loc="upper right", fontsize=8, framealpha=0.8)

    axes[-1].set_xlabel("Time from first data point (s)")
    fig.suptitle(f"Session Timeline — {test['id']}", fontweight="bold")
    plt.show()